In [1]:
# ============================================================
# ATTACK-TYPE CLASSIFIER (FLOW / LAYER-1)
# StratifiedKFold, leakage-safe, memory-optimized
#
# Veri kaynaklari:
#   1. CICIDS2017 saldirilari
#   2. CSE-CIC-IDS2018 Infiltration
#   3. CSE-CIC-IDS2018 SQLi/XSS/BruteForce (opsiyonel ek kaynak
#      - bkz. bolum 2'deki CSE_CIC_IDS_2018_WEBATTACK_FILES)
#
# Hiyerarsik web saldirisi yaklasimi:
#
#   CICIDS2017:
#       Web Attack - XSS
#       Web Attack - Sql Injection
#       Web Attack - Brute Force
#
#   CSE-CIC-IDS2018 (opsiyonel ek kaynak):
#       SQL Injection
#       Brute Force -Web
#       Brute Force -XSS
#
#   Flow modelinde tek sinifa donusturulur:
#       WebAttackCandidate
#
#   WebAttackCandidate tahmin edilen flow:
#       HTTP payload analiz katmanina yonlendirilir
#       (bkz. payload_classifier_csic2010.py - Layer 2).
#
#   Kesin ayrim ikinci katmanda (CSIC2010 tabanli payload
#   classifier) yapilir:
#       SQL Injection
#       XSS
#       Web Brute Force  (payload SQLi/XSS ile eslesmezse,
#                          eleme yontemiyle bu sinifa dusuyor
#                          - cunku WebAttackCandidate zaten
#                          yalnizca bu 3 alt turden birini
#                          temsil ediyor)
#
# ONEMLI - SQLI VERI AZLIGI VE COZUMU HAKKINDA:
#   CICIDS2017 icindeki "Web Attack - Sql Injection" alt sinifi
#   yalnizca ~21 kayittan olusuyor. Bu, WebAttackCandidate
#   sinifinin kendi icinde bile ciddi bir dengesizlik yaratiyor
#   ve OOF degerlendirmesinde SQLi-orijinli flow'lar icin
#   routing recall'un (dogru sekilde WebAttackCandidate olarak
#   isaretlenip Layer 2'ye yonlendirilme orani) diger alt
#   turlere (Brute Force, XSS) gore belirgin dusuk cikmasina
#   yol aciyor.
#
#   BU NEDENLE: CSE-CIC-IDS2018'in ayni CICFlowMeter 80-kolon
#   semasina sahip, SQLi/XSS/BruteForce icin AYRI dosyalar
#   sunan versiyonu (bkz. Mendeley temiz surumu) destekleniyor.
#   Bu dosyalar mevcutsa (data/ klasorune indirilmisse) otomatik
#   olarak yuklenir ve WebAttackCandidate sinifina eklenir; hic
#   kod degisikligi gerekmez (normalize_label zaten bu
#   etiketleri taniyor). Dosyalar yoksa sessizce atlanir.
#
# ONEMLI - HEARTBLEED HAKKINDA:
#   Heartbleed, CICIDS2017 icinde
#   "Wednesday-workingHours.pcap_ISCX.csv" dosyasinda bulunur
#   ve dogal olarak cok az orneklidir (tipik olarak ~11 kayit).
#
#   Bu sinif istatistiksel olarak anlamli bir CV degerlendirmesi
#   icin yeterli degildir ve StratifiedKFold fold sayisini
#   (N_SPLITS) TUM veri seti icin gereksiz yere dusuruyordu.
#
#   BU NEDENLE: Heartbleed kayitlari bu projede degerlendirme
#   disi birakilmistir. Kayitlar, CSV okuma asamasinda
#   (load_and_prepare_csv icinde) kaynaginda tamamen
#   cikarilir; ne egitimde, ne CV'de, ne de nihai modelde
#   yer alir. Bu davranis calisma zamaninda loglanir.
#
# ONEMLI - INFILTRATION HAKKINDA:
#   data/Wednesday-28-02-2018.csv bu projede yalnizca
#   Infiltration kaynagi olarak kullanilmaktadir.
# ============================================================

import os
import re
import gc
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)


def log(*args):
    """Mesajlari aninda terminale yazdirir."""
    print(*args, flush=True)


# ============================================================
# 1. GENEL AYARLAR
# ============================================================

RANDOM_STATE = 42

N_SPLITS_MAX = 5
N_SPLITS_MIN = 2

CV_N_ESTIMATORS = 200
FINAL_N_ESTIMATORS = 500

# M4 MacBook uzerinde RAM kullanimini sinirlamak icin.
N_JOBS = 4

MIN_CLASS_SAMPLES = 10

WEB_ATTACK_CANDIDATE_LABEL = "WebAttackCandidate"

# WebAttackCandidate tahmini bu degerin altindaysa da kayit
# kaybolmaz. Manuel incelemeye veya payload analizine gider.
WEB_CANDIDATE_CONFIDENCE_THRESHOLD = 0.50

# Genel olarak "az orneli" (rare) siniflari loglamak icin
# kullanilan uyari esigi. Bu, veriyi silmez, sadece bilgi
# amacli raporlar.
RARE_CLASS_WARNING_THRESHOLD = 20


# ============================================================
# 2. DOSYA VE KLASOR AYARLARI
# ============================================================

DATA_DIR = "data"
MODEL_DIR = "models"
OUTPUT_DIR = "outputs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


CICIDS_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",

    # Bu dosya Heartbleed kayitlarini da icerir; bu kayitlar
    # load_and_prepare_csv icinde otomatik olarak cikarilir.
    "Wednesday-workingHours.pcap_ISCX.csv",

    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]


# Bu dosya senin projendeki Infiltration dosyasidir.
# Web attack dosyasi olarak degistirilmemektedir.
CSE_CIC_IDS_2018_INFILTRATION_FILE = os.path.join(
    DATA_DIR,
    "Wednesday-28-02-2018.csv"
)

# ------------------------------------------------------------
# CSE-CIC-IDS2018 - EK WEB ATTACK (SQLi/XSS/BruteForce) DOSYALARI
# ------------------------------------------------------------
# CICIDS2017'deki WebAttackCandidate sinifinin icinde SQL
# Injection orijinli kayit sayisi son derece az (~21 kayit).
# Bu, hem flow modelinin bu alt-paterni yeterince ogrenememesine
# hem de OOF degerlendirmesinde SQLi-orijinli kayitlar icin
# routing recall'un dusuk cikmasina (istatistiksel gurultuye
# de acik, ama esas neden veri azligi) yol aciyor.
#
# CSE-CIC-IDS2018 dataseti (ayni CICFlowMeter 80-kolon semasi
# ile), SQLi/XSS/BruteForce icin AYRI dosyalar halinde ek flow
# kayitlari sunuyor (Mendeley'in temizlenmis surumu -
# https://data.mendeley.com/datasets/29hdbdzx2r/1):
#
#   SQL Injection.csv
#   Brute Force -Web.csv
#   Brute Force -XSS.csv
#
# Bu dosyalar mevcutsa (data/ klasorune indirilmisse), asagida
# otomatik olarak yuklenir ve normalize_label() zaten bu
# etiketleri WebAttackCandidate'e cevirdigi icin, ekstra kod
# degisikligi gerekmeden mevcut egitim/CV akisina dahil olur.
#
# Dosya bulunamazsa sessizce atlanir (UYARI loglanir) - yani
# bu dosyalari indirmeden de script calismaya devam eder.
#
# NOT (gun/tarih karmasasi): CSE-CIC-IDS2018'in gun -> dosya
# eslemesi kaynaklar arasinda tutarsizdir (bazi mirror'larda
# "28 Subat" Web Attacks gunudur, bazilarinda Infiltration
# gunudur). Bu belirsizligi tamamen bypass etmek icin, gun
# bazli birlesik dosyalar yerine Mendeley'in SINIF BAZINDA
# ayirdigi dosyalar tercih edilmistir.
CSE_CIC_IDS_2018_WEBATTACK_FILES = [
    os.path.join(DATA_DIR, "SQL Injection.csv"),
    os.path.join(DATA_DIR, "Brute Force -Web.csv"),
    os.path.join(DATA_DIR, "Brute Force -XSS.csv"),
]


MODEL_PATH = os.path.join(
    MODEL_DIR,
    "attack_classifier.joblib"
)

REPORT_PATH = os.path.join(
    OUTPUT_DIR,
    "attack_classification_report_oof.csv"
)

FOLD_METRICS_PATH = os.path.join(
    OUTPUT_DIR,
    "attack_fold_metrics.csv"
)

CONFUSION_MATRIX_PATH = os.path.join(
    OUTPUT_DIR,
    "attack_confusion_matrix_oof.png"
)

FEATURE_IMPORTANCE_PATH = os.path.join(
    OUTPUT_DIR,
    "attack_feature_importance.csv"
)

DATASET_SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "attack_dataset_summary.csv"
)

PREDICTIONS_PATH = os.path.join(
    OUTPUT_DIR,
    "attack_oof_predictions.csv"
)

INFILTRATION_BY_SOURCE_PATH = os.path.join(
    OUTPUT_DIR,
    "infiltration_performance_by_dataset.csv"
)

WEB_CANDIDATE_ANALYSIS_PATH = os.path.join(
    OUTPUT_DIR,
    "webattackcandidate_oof_analysis.csv"
)

WEB_CANDIDATE_BY_ORIGINAL_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "webattackcandidate_performance_by_original_label.csv"
)

RARE_CLASS_SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "rare_class_summary.csv"
)


# ============================================================
# 3. ORTAK FEATURE SEMASI
# ============================================================

REQUESTED_FEATURES = [
    "Destination Port",
    "Flow Duration",

    "Total Fwd Packets",
    "Total Backward Packets",

    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",

    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",

    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",

    "Flow Bytes/s",
    "Flow Packets/s",

    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",

    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",

    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",

    "Fwd PSH Flags",
    "Bwd PSH Flags",

    "Fwd URG Flags",
    "Bwd URG Flags",

    "Fwd Header Length",
    "Bwd Header Length",

    "Fwd Packets/s",
    "Bwd Packets/s",

    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "Packet Length Variance",

    "FIN Flag Count",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "CWE Flag Count",
    "ECE Flag Count",

    "Down/Up Ratio",
    "Average Packet Size",

    "Avg Fwd Segment Size",
    "Avg Bwd Segment Size",

    "Fwd Header Length.1",

    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",

    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",

    "Subflow Fwd Packets",
    "Subflow Fwd Bytes",

    "Subflow Bwd Packets",
    "Subflow Bwd Bytes",

    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",

    "act_data_pkt_fwd",
    "min_seg_size_forward",

    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",

    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min"
]


# ============================================================
# 4. CSE-CIC-IDS2018 KOLONLARINI CICIDS2017 SEMASINA CEVIR
# ============================================================

COLUMN_MAPPING_2018_TO_2017 = {
    "Dst Port": "Destination Port",

    "Tot Fwd Pkts": "Total Fwd Packets",
    "Tot Bwd Pkts": "Total Backward Packets",

    "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets",

    "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min",
    "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std",

    "Bwd Pkt Len Max": "Bwd Packet Length Max",
    "Bwd Pkt Len Min": "Bwd Packet Length Min",
    "Bwd Pkt Len Mean": "Bwd Packet Length Mean",
    "Bwd Pkt Len Std": "Bwd Packet Length Std",

    "Flow Byts/s": "Flow Bytes/s",
    "Flow Pkts/s": "Flow Packets/s",

    "Fwd IAT Tot": "Fwd IAT Total",
    "Bwd IAT Tot": "Bwd IAT Total",

    "Fwd Header Len": "Fwd Header Length",
    "Bwd Header Len": "Bwd Header Length",

    "Fwd Pkts/s": "Fwd Packets/s",
    "Bwd Pkts/s": "Bwd Packets/s",

    "Pkt Len Min": "Min Packet Length",
    "Pkt Len Max": "Max Packet Length",
    "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std",
    "Pkt Len Var": "Packet Length Variance",

    "FIN Flag Cnt": "FIN Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "CWE Flag Count": "CWE Flag Count",
    "ECE Flag Cnt": "ECE Flag Count",

    "Pkt Size Avg": "Average Packet Size",

    "Fwd Seg Size Avg": "Avg Fwd Segment Size",
    "Bwd Seg Size Avg": "Avg Bwd Segment Size",

    "Fwd Byts/b Avg": "Fwd Avg Bytes/Bulk",
    "Fwd Pkts/b Avg": "Fwd Avg Packets/Bulk",
    "Fwd Blk Rate Avg": "Fwd Avg Bulk Rate",

    "Bwd Byts/b Avg": "Bwd Avg Bytes/Bulk",
    "Bwd Pkts/b Avg": "Bwd Avg Packets/Bulk",
    "Bwd Blk Rate Avg": "Bwd Avg Bulk Rate",

    "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Fwd Byts": "Subflow Fwd Bytes",

    "Subflow Bwd Pkts": "Subflow Bwd Packets",
    "Subflow Bwd Byts": "Subflow Bwd Bytes",

    "Init Fwd Win Byts": "Init_Win_bytes_forward",
    "Init Bwd Win Byts": "Init_Win_bytes_backward",

    "Fwd Act Data Pkts": "act_data_pkt_fwd",
    "Fwd Seg Size Min": "min_seg_size_forward"
}


# ============================================================
# 5. LABEL NORMALIZASYONU
# ============================================================

def normalize_label(label):
    """
    Farkli datasetlerdeki label degerlerini ortak isimlere
    donusturur.

    XSS, SQL Injection ve Web Brute Force ayri flow siniflari
    olarak tutulmaz. Ucu de WebAttackCandidate sinifina
    donusturulur. Kesin web saldirisi turu, bu script ile
    birlikte calisan payload_classifier_csic2010.py (Layer 2)
    tarafindan CSIC2010 HTTP payload'lari uzerinden belirlenir.

    Bu fonksiyon hem CICIDS2017'nin "Web Attack - Sql
    Injection" tarzi etiketlerini, hem de CSE-CIC-IDS2018'in
    ayri dosyalarindaki "SQL Injection", "Brute Force -Web",
    "Brute Force -XSS" etiketlerini ayni sekilde
    WebAttackCandidate'e cevirir - boylece iki farkli
    datasetten gelen web saldirisi kayitlari sorunsuzca
    birlestirilebilir.

    NOT: Heartbleed bu fonksiyonda ayri bir sinif olarak
    ELE ALINMAZ. Heartbleed kayitlari load_and_prepare_csv
    icinde, bu fonksiyon cagrilmadan hemen once/sonra,
    kaynaginda tamamen filtrelenip atilir (bkz. asagidaki
    load_and_prepare_csv). Boylece Heartbleed veri setine
    hic girmez ve degerlendirmeyi etkilemez.
    """

    label = str(label).strip()
    label = re.sub(r"\s+", " ", label)

    label = (
        label
        .replace("ï¿½", "-")
        .replace("�", "-")
        .replace("–", "-")
        .replace("—", "-")
    )

    lower_label = label.casefold()

    # Bazi CSV dosyalarinda header satiri veri icinde tekrar eder.
    if lower_label == "label":
        return "HEADER_ROW"

    if lower_label == "benign":
        return "BENIGN"

    # --------------------------------------------------------
    # WEB ATTACK CANDIDATE
    # --------------------------------------------------------

    # CICIDS2017 ornekleri:
    # Web Attack - Brute Force
    # Web Attack - XSS
    # Web Attack - Sql Injection

    if "web attack" in lower_label:
        return WEB_ATTACK_CANDIDATE_LABEL

    # CSE-CIC-IDS2018'in ayri dosyalarindaki etiketler:
    # SQL Injection
    # Brute Force -Web
    # Brute Force -XSS
    explicit_web_labels = {
        "brute force -web",
        "brute force - web",
        "brute force -xss",
        "brute force - xss",
        "sql injection",
        "sql-injection",
        "sqli",
        "xss",
        "cross site scripting",
        "cross-site scripting"
    }

    if lower_label in explicit_web_labels:
        return WEB_ATTACK_CANDIDATE_LABEL

    compact_label = re.sub(
        r"[\s_\-]+",
        "",
        lower_label
    )

    compact_web_labels = {
        "webattackbruteforce",
        "webattackxss",
        "webattacksqlinjection",
        "bruteforceweb",
        "bruteforcexss",
        "sqlinjection",
        "sqli",
        "xss",
        "crosssitescripting"
    }

    if compact_label in compact_web_labels:
        return WEB_ATTACK_CANDIDATE_LABEL

    # --------------------------------------------------------
    # INFILTRATION
    # --------------------------------------------------------

    if lower_label in {
        "infilteration",
        "infiltration"
    }:
        return "Infiltration"

    # --------------------------------------------------------
    # FTP VE SSH BRUTE FORCE
    # --------------------------------------------------------

    if (
        "ftp" in lower_label
        and "patator" in lower_label
    ):
        return "FTP-Patator"

    if (
        "ssh" in lower_label
        and "patator" in lower_label
    ):
        return "SSH-Patator"

    if (
        "ftp" in lower_label
        and "brute" in lower_label
    ):
        return "FTP-BruteForce"

    if (
        "ssh" in lower_label
        and "brute" in lower_label
    ):
        return "SSH-BruteForce"

    # --------------------------------------------------------
    # BOTNET
    # --------------------------------------------------------

    if lower_label in {
        "bot",
        "botnet"
    }:
        return "Botnet"

    # --------------------------------------------------------
    # PORTSCAN
    # --------------------------------------------------------

    if lower_label in {
        "portscan",
        "port scan"
    }:
        return "PortScan"

    # --------------------------------------------------------
    # DDOS
    # --------------------------------------------------------

    if lower_label == "ddos":
        return "DDoS"

    if "ddos" in lower_label:

        if "hoic" in lower_label:
            return "DDoS-HOIC"

        if (
            "loic" in lower_label
            and "udp" in lower_label
        ):
            return "DDoS-LOIC-UDP"

        if "loic" in lower_label:
            return "DDoS-LOIC-HTTP"

        return "DDoS"

    # --------------------------------------------------------
    # DOS
    # --------------------------------------------------------

    if "slowhttptest" in lower_label:
        return "DoS Slowhttptest"

    if "slowloris" in lower_label:
        return "DoS slowloris"

    if "goldeneye" in lower_label:
        return "DoS GoldenEye"

    if "hulk" in lower_label:
        return "DoS Hulk"

    # NOT: "heartbleed" buraya kasitli olarak dahil edilmedi.
    # Heartbleed kayitlari zaten load_and_prepare_csv icinde
    # bu fonksiyon calismadan once/sonra veri setinden
    # cikarilir (asagiya bakiniz). Eger bu filtre herhangi
    # bir nedenle atlanirsa, "Heartbleed" etiketi asagidaki
    # "return label" satirina dusup orijinal haliyle
    # korunur - bu da calisma zamaninda kolayca fark edilir.

    # Taninmayan etiket orijinal haliyle korunur.
    return label


# ============================================================
# 6. CSV OKUMA VE TEMIZLEME
# ============================================================

def load_and_prepare_csv(file_path, dataset_name):
    """
    CSV dosyasini okur, kolonlari normalize eder, label
    donusumunu yapar ve yalnizca saldiri kayitlarini dondurur.

    Heartbleed kayitlari bu fonksiyon icinde kaynaginda
    tamamen cikarilir (asagidaki adim 6.1'e bakiniz).
    """

    log("\n" + "=" * 80)
    log("Dosya okunuyor:", file_path)
    log("Dataset:", dataset_name)

    if not os.path.exists(file_path):
        log("UYARI: Dosya bulunamadi, atlandi.")
        return None

    current_df = pd.read_csv(
        file_path,
        low_memory=False
    )

    log("Ham boyut:", current_df.shape)

    current_df.columns = (
        current_df.columns
        .astype(str)
        .str.strip()
    )

    if "Label" not in current_df.columns:
        log("UYARI: Label kolonu bulunamadi, atlandi.")
        log("Kolonlar:", current_df.columns.tolist())
        return None

    # 2018 kisa kolonlarini 2017 semasina cevir.
    applicable_mapping = {
        old_name: new_name
        for old_name, new_name
        in COLUMN_MAPPING_2018_TO_2017.items()
        if (
            old_name in current_df.columns
            and new_name not in current_df.columns
        )
    }

    if applicable_mapping:
        current_df.rename(
            columns=applicable_mapping,
            inplace=True
        )

        log(
            "Eslenen 2018 kolon sayisi:",
            len(applicable_mapping)
        )

    # Orijinal label audit ve analiz icin korunur.
    current_df["_original_label"] = (
        current_df["Label"]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # 6.1 HEARTBLEED'I KAYNAGINDA CIKAR
    # --------------------------------------------------------
    # Heartbleed bu projede degerlendirme disi birakildi
    # (dogal olarak ~11 kayitla anlamli bir CV degerlendirmesi
    # yapilamiyor ve StratifiedKFold fold sayisini tum veri
    # seti icin gereksiz yere dusuruyordu). Bu nedenle
    # Heartbleed kayitlari, Label donusumunden once, orijinal
    # etiket uzerinden tespit edilip veri setinden tamamen
    # cikarilir. Egitime, CV'ye ve nihai modele hic girmez.
    heartbleed_mask = (
        current_df["_original_label"]
        .str.contains("heartbleed", case=False, na=False)
    )

    if heartbleed_mask.any():

        log(
            "\nHeartbleed kaydi bulundu ve degerlendirme "
            "disi birakildi. Cikarilan kayit sayisi:",
            int(heartbleed_mask.sum())
        )

        current_df = current_df[~heartbleed_mask].copy()

    # Modelin kullanacagi normalize edilmis label.
    current_df["Label"] = (
        current_df["Label"]
        .apply(normalize_label)
    )

    # CSV icinde tekrar eden header satirlarini kaldir.
    current_df = current_df[
        current_df["Label"] != "HEADER_ROW"
    ].copy()

    log("\nBENIGN filtresi oncesi siniflar:")
    log(
        current_df["Label"]
        .value_counts()
        .head(30)
    )

    # Bu model Morpheus sonrasinda calisacagi icin yalnizca
    # saldiri turlerini ogrenir. BENIGN kaldirilir.
    current_df = current_df[
        current_df["Label"] != "BENIGN"
    ].copy()

    if current_df.empty:
        log("Bu dosyada saldiri kaydi bulunamadi.")
        return None

    current_df["_source_file"] = Path(file_path).name
    current_df["_dataset_name"] = dataset_name

    current_df["_requires_payload_analysis"] = (
        current_df["Label"]
        == WEB_ATTACK_CANDIDATE_LABEL
    )

    # flow_id mevcut degilse guvenilir bir benzersiz ID uret.
    if "flow_id" not in current_df.columns:

        current_df["flow_id"] = [
            (
                f"{dataset_name}:"
                f"{Path(file_path).name}:"
                f"{index}"
            )
            for index in current_df.index
        ]

    else:

        missing_flow_id_mask = (
            current_df["flow_id"].isna()
            | (
                current_df["flow_id"]
                .astype(str)
                .str.strip()
                == ""
            )
        )

        generated_ids = np.array(
            [
                (
                    f"{dataset_name}:"
                    f"{Path(file_path).name}:"
                    f"{index}"
                )
                for index in current_df.index
            ],
            dtype=object
        )

        current_df.loc[
            missing_flow_id_mask,
            "flow_id"
        ] = generated_ids[
            missing_flow_id_mask.to_numpy()
        ]

    log("\nKalan saldiri kayitlari:")
    log(current_df["Label"].value_counts())

    if (
        WEB_ATTACK_CANDIDATE_LABEL
        in current_df["Label"].unique()
    ):
        log("\nWebAttackCandidate orijinal dagilimi:")

        log(
            current_df.loc[
                current_df["Label"]
                == WEB_ATTACK_CANDIDATE_LABEL,
                "_original_label"
            ].value_counts()
        )

    return current_df


# ============================================================
# 7. CICIDS2017 DOSYALARINI YUKLE
# ============================================================

attack_frames = []

for file_name in CICIDS_FILES:

    file_path = os.path.join(
        DATA_DIR,
        file_name
    )

    prepared_df = load_and_prepare_csv(
        file_path=file_path,
        dataset_name="CICIDS2017"
    )

    if prepared_df is not None:
        attack_frames.append(prepared_df)


# ============================================================
# 8. CSE-CIC-IDS2018 INFILTRATION DOSYASINI YUKLE
# ============================================================

infiltration_2018_df = load_and_prepare_csv(
    file_path=CSE_CIC_IDS_2018_INFILTRATION_FILE,
    dataset_name="CSE-CIC-IDS2018"
)

if infiltration_2018_df is not None:

    # Bu dosyadan yalnizca Infiltration kayitlari alinir.
    infiltration_2018_df = infiltration_2018_df[
        infiltration_2018_df["Label"]
        == "Infiltration"
    ].copy()

    log(
        "\nCSE-CIC-IDS2018 dosyasindan "
        "eklenecek Infiltration:"
    )

    log(
        infiltration_2018_df["Label"]
        .value_counts()
    )

    if not infiltration_2018_df.empty:
        attack_frames.append(infiltration_2018_df)

    else:
        log(
            "UYARI: Infiltration dosyasi okundu fakat "
            "normalize edilmis Infiltration satiri bulunamadi."
        )


# ============================================================
# 8B. CSE-CIC-IDS2018 EK WEB ATTACK (SQLi/XSS/BruteForce)
#     DOSYALARINI YUKLE
# ============================================================
#
# CICIDS2017'deki SQLi orijinli kayit sayisi (~21) hem egitim
# hem de OOF degerlendirme acisindan yetersiz kaliyor ve bu
# alt-turun routing recall'unu dusuruyor. Bu blok, mevcutsa
# CSE_CIC_IDS_2018_WEBATTACK_FILES icindeki dosyalari yukleyip
# WebAttackCandidate sinifina ekler.
#
# normalize_label() zaten "SQL Injection", "Brute Force -Web",
# "Brute Force -XSS" etiketlerini WebAttackCandidate'e
# cevirdigi icin ekstra bir esleme kodu gerekmez.
#
# Dosyalar bulunamazsa (indirilmemisse) bu blok sessizce
# atlanir ve pipeline geri kalaniyla calismaya devam eder.

webattack_2018_added_count = 0

for webattack_file_path in CSE_CIC_IDS_2018_WEBATTACK_FILES:

    webattack_2018_df = load_and_prepare_csv(
        file_path=webattack_file_path,
        dataset_name="CSE-CIC-IDS2018-WebAttacks"
    )

    if webattack_2018_df is None:
        continue

    # Guvenlik amacli: bu dosyalardan yalnizca
    # WebAttackCandidate'e donusen kayitlar alinir. Beklenmedik
    # bir etiket (ornegin yanlislikla farkli bir saldiri turu)
    # varsa dahil edilmez.
    webattack_2018_df = webattack_2018_df[
        webattack_2018_df["Label"]
        == WEB_ATTACK_CANDIDATE_LABEL
    ].copy()

    if webattack_2018_df.empty:
        log(
            "\nUYARI:", webattack_file_path,
            "okundu fakat WebAttackCandidate satiri "
            "bulunamadi (etiket eslemesini kontrol et)."
        )
        continue

    log(
        f"\n{Path(webattack_file_path).name} dosyasindan "
        f"eklenecek WebAttackCandidate ({len(webattack_2018_df)} "
        f"kayit):"
    )

    log(
        webattack_2018_df["_original_label"]
        .value_counts()
    )

    attack_frames.append(webattack_2018_df)
    webattack_2018_added_count += len(webattack_2018_df)


if webattack_2018_added_count > 0:
    log(
        "\nCSE-CIC-IDS2018'den WebAttackCandidate sinifina "
        f"eklenen toplam ek kayit: {webattack_2018_added_count}"
    )
else:
    log(
        "\nCSE-CIC-IDS2018 web attack dosyalari bulunamadi "
        "(SQL Injection.csv / Brute Force -Web.csv / "
        "Brute Force -XSS.csv). Yalnizca CICIDS2017'deki "
        "~21 SQLi ornegiyle devam edilecek. Bu dosyalari "
        f"'{DATA_DIR}/' klasorune eklersen (Mendeley'in "
        "temizlenmis CSE-CIC-IDS2018 surumunden) SQLi "
        "routing recall'u iyilesecektir."
    )


# ============================================================
# 9. TUM SALDIRI VERILERINI BIRLESTIR
# ============================================================

if not attack_frames:
    raise ValueError(
        "Hicbir saldiri dataseti yuklenemedi. "
        "DATA_DIR ve dosya yollarini kontrol et."
    )

attack_df = pd.concat(
    attack_frames,
    ignore_index=True,
    sort=False
)

del attack_frames
gc.collect()


log("\n" + "=" * 80)
log("BIRLESTIRILMIS SALDIRI DATASETI")
log("=" * 80)

log("Toplam saldiri satiri:", len(attack_df))
log("Toplam kolon:", len(attack_df.columns))

log("\nSinif dagilimi:")
log(attack_df["Label"].value_counts())

log("\nDataset bazinda dagilim:")
log(
    pd.crosstab(
        attack_df["_dataset_name"],
        attack_df["Label"]
    )
)


# ============================================================
# 10. WEBATTACKCANDIDATE VERI KONTROLU
# ============================================================

web_candidate_mask_initial = (
    attack_df["Label"]
    == WEB_ATTACK_CANDIDATE_LABEL
)

if web_candidate_mask_initial.any():

    log("\n" + "=" * 80)
    log("WEBATTACKCANDIDATE VERI KONTROLU")
    log("=" * 80)

    log(
        "Toplam WebAttackCandidate:",
        int(web_candidate_mask_initial.sum())
    )

    log("\nDataset bazinda WebAttackCandidate:")

    log(
        attack_df.loc[
            web_candidate_mask_initial,
            "_dataset_name"
        ].value_counts()
    )

    log("\nOrijinal web attack etiketleri:")

    log(
        attack_df.loc[
            web_candidate_mask_initial,
            "_original_label"
        ].value_counts()
    )

else:
    log(
        "\nUYARI: WebAttackCandidate bulunamadi. "
        "CICIDS2017 WebAttacks CSV dosyasini ve "
        "Label degerlerini kontrol et."
    )


# ============================================================
# 11. INFILTRATION VERI KONTROLU
# ============================================================

if "Infiltration" in attack_df["Label"].unique():

    infiltration_data = attack_df[
        attack_df["Label"] == "Infiltration"
    ]

    log("\nInfiltration kaynak dagilimi:")

    log(
        infiltration_data[
            "_dataset_name"
        ].value_counts()
    )

    log("\nInfiltration dosya dagilimi:")

    log(
        infiltration_data[
            "_source_file"
        ].value_counts()
    )


# ============================================================
# 12. COK AZ KAYITLI SINIFLARI KALDIR
# ============================================================

class_counts = attack_df["Label"].value_counts()

too_small_classes = class_counts[
    class_counts < MIN_CLASS_SAMPLES
].index.tolist()

if too_small_classes:

    log(
        "\nEgitim icin cok az kayitli siniflar "
        "kaldiriliyor:",
        too_small_classes
    )

    attack_df = attack_df[
        ~attack_df["Label"].isin(
            too_small_classes
        )
    ].copy()


log("\nTemizlik sonrasi sinif dagilimi:")
log(attack_df["Label"].value_counts())


# ============================================================
# 13. KULLANILABILIR FEATURE'LARI SEC
# ============================================================

FEATURES = [
    feature
    for feature in REQUESTED_FEATURES
    if feature in attack_df.columns
]

MISSING_FEATURES = [
    feature
    for feature in REQUESTED_FEATURES
    if feature not in attack_df.columns
]


log("\nKullanilacak feature sayisi:", len(FEATURES))

log("\nKullanilacak feature'lar:")

for feature in FEATURES:
    log("  +", feature)


if MISSING_FEATURES:

    log(
        "\nCSV'lerde bulunamayan ve atlanan "
        "feature'lar:"
    )

    for feature in MISSING_FEATURES:
        log("  -", feature)


if not FEATURES:
    raise ValueError(
        "Model egitimi icin kullanilabilir "
        "feature bulunamadi."
    )


# ============================================================
# 14. X, Y VE METADATA HAZIRLA
# ============================================================

X_all = attack_df[FEATURES].copy()
y_all = attack_df["Label"].copy()

flow_ids_all = attack_df["flow_id"].copy()
source_files_all = attack_df["_source_file"].copy()
dataset_names_all = attack_df["_dataset_name"].copy()
original_labels_all = attack_df["_original_label"].copy()

requires_payload_analysis_all = attack_df[
    "_requires_payload_analysis"
].copy()


# ============================================================
# 15. SAYISAL DONUSUM, INF VE NAN TEMIZLIGI
# ============================================================

for column in X_all.columns:

    X_all[column] = pd.to_numeric(
        X_all[column],
        errors="coerce"
    ).astype("float32")


X_all.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


all_nan_columns = X_all.columns[
    X_all.isna().all()
].tolist()

if all_nan_columns:

    log(
        "\nTamamen NaN oldugu icin kaldirilan "
        "feature'lar:"
    )

    log(all_nan_columns)

    X_all.drop(
        columns=all_nan_columns,
        inplace=True
    )


constant_columns = [
    column
    for column in X_all.columns
    if X_all[column].nunique(dropna=True) <= 1
]

if constant_columns:

    log(
        "\nSabit oldugu icin kaldirilan "
        "feature sayisi:",
        len(constant_columns)
    )

    log(constant_columns)

    X_all.drop(
        columns=constant_columns,
        inplace=True
    )


FEATURES = X_all.columns.tolist()

log("\nSon X sekli:", X_all.shape)
log("y sekli:", y_all.shape)
log("Son kullanilabilir feature:", len(FEATURES))


# ============================================================
# 16. DUPLICATE TEMIZLIGI
# ============================================================

combined_for_duplicates = X_all.copy()

combined_for_duplicates["_label"] = y_all.to_numpy()
combined_for_duplicates["_flow_id"] = flow_ids_all.to_numpy()

combined_for_duplicates["_source_file"] = (
    source_files_all.to_numpy()
)

combined_for_duplicates["_dataset_name"] = (
    dataset_names_all.to_numpy()
)

combined_for_duplicates["_original_label"] = (
    original_labels_all.to_numpy()
)

combined_for_duplicates[
    "_requires_payload_analysis"
] = requires_payload_analysis_all.to_numpy()


duplicate_subset = FEATURES + ["_label"]

before_duplicates = len(combined_for_duplicates)

combined_for_duplicates = (
    combined_for_duplicates
    .drop_duplicates(
        subset=duplicate_subset,
        keep="first"
    )
    .copy()
)

removed_duplicates = (
    before_duplicates
    - len(combined_for_duplicates)
)

log(
    "\nSilinen duplicate saldiri satiri:",
    removed_duplicates
)


y_all = combined_for_duplicates.pop("_label")
flow_ids_all = combined_for_duplicates.pop("_flow_id")

source_files_all = combined_for_duplicates.pop(
    "_source_file"
)

dataset_names_all = combined_for_duplicates.pop(
    "_dataset_name"
)

original_labels_all = combined_for_duplicates.pop(
    "_original_label"
)

requires_payload_analysis_all = (
    combined_for_duplicates.pop(
        "_requires_payload_analysis"
    )
)

X_all = combined_for_duplicates.copy()

del combined_for_duplicates
gc.collect()


X_all = X_all.reset_index(drop=True)
y_all = y_all.reset_index(drop=True)

flow_ids_all = flow_ids_all.reset_index(drop=True)
source_files_all = source_files_all.reset_index(drop=True)
dataset_names_all = dataset_names_all.reset_index(drop=True)
original_labels_all = original_labels_all.reset_index(drop=True)

requires_payload_analysis_all = (
    requires_payload_analysis_all
    .reset_index(drop=True)
)


log("Duplicate sonrasi X:", X_all.shape)

log("\nDuplicate sonrasi sinif dagilimi:")
log(y_all.value_counts())


# ============================================================
# 17. NADIR SINIF OZETI
# ============================================================

rare_class_counts = y_all.value_counts()

rare_class_summary = rare_class_counts[
    rare_class_counts < RARE_CLASS_WARNING_THRESHOLD
]

if not rare_class_summary.empty:

    log("\n" + "=" * 80)
    log("NADIR SINIF OZETI")
    log("=" * 80)

    log(
        f"Su siniflar {RARE_CLASS_WARNING_THRESHOLD} "
        "kayittan az ornege sahip:"
    )

    log(rare_class_summary)

    rare_class_summary.rename("sample_count").to_csv(
        RARE_CLASS_SUMMARY_PATH
    )


# ============================================================
# 18. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(
    y_all
)


log("\nAttack siniflari:")

for class_index, class_name in enumerate(
    label_encoder.classes_
):
    log(class_index, "->", class_name)


if (
    WEB_ATTACK_CANDIDATE_LABEL
    not in label_encoder.classes_
):
    log(
        "\nUYARI: Egitim siniflarinda "
        "WebAttackCandidate bulunmuyor."
    )


# ============================================================
# 19. STRATIFIEDKFOLD KURULUMU
# ============================================================

class_counts_final = (
    pd.Series(y_encoded)
    .value_counts()
)

smallest_class_count = int(
    class_counts_final.min()
)

smallest_class_encoded = int(
    class_counts_final.idxmin()
)

smallest_class_name = label_encoder.inverse_transform(
    [smallest_class_encoded]
)[0]

N_SPLITS = min(
    N_SPLITS_MAX,
    smallest_class_count
)

if N_SPLITS < N_SPLITS_MIN:

    raise ValueError(
        f"En az orneli sinif '{smallest_class_name}' "
        f"icinde {smallest_class_count} kayit var. "
        f"StratifiedKFold icin en az "
        f"{N_SPLITS_MIN} gerekir."
    )


log(
    "\nEn az orneli sinif:",
    smallest_class_name,
    "| boyutu:",
    smallest_class_count
)

log(
    "Kullanilacak fold sayisi:",
    N_SPLITS
)


skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


CV_MODEL_PARAMS = {
    "n_estimators": CV_N_ESTIMATORS,
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS
}


FINAL_MODEL_PARAMS = {
    "n_estimators": FINAL_N_ESTIMATORS,
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS
}


# ============================================================
# 20. STRATIFIEDKFOLD CROSS-VALIDATION
# ============================================================

n_rows = len(X_all)

oof_pred_encoded = np.full(
    shape=n_rows,
    fill_value=-1,
    dtype=int
)

oof_confidence = np.full(
    shape=n_rows,
    fill_value=np.nan,
    dtype=float
)

fold_metrics_records = []


log("\n" + "=" * 80)

log(
    f"STRATIFIEDKFOLD BASLIYOR "
    f"({N_SPLITS} fold, "
    f"n_estimators={CV_N_ESTIMATORS}, "
    f"n_jobs={N_JOBS})"
)

log("=" * 80)


for fold_index, (train_idx, test_idx) in enumerate(
    skf.split(X_all, y_encoded),
    start=1
):

    log(
        f"\n[Fold {fold_index}/{N_SPLITS}] "
        f"basliyor..."
    )

    X_train_fold = X_all.iloc[train_idx]
    X_test_fold = X_all.iloc[test_idx]

    y_train_fold = y_encoded[train_idx]
    y_test_fold = y_encoded[test_idx]

    # Imputer yalnizca train fold uzerinde fit edilir.
    # Boylece test fold bilgisi egitime sizmaz.
    fold_imputer = SimpleImputer(
        strategy="median"
    )

    X_train_fold_clean = (
        fold_imputer
        .fit_transform(X_train_fold)
        .astype("float32")
    )

    X_test_fold_clean = (
        fold_imputer
        .transform(X_test_fold)
        .astype("float32")
    )

    fold_model = ExtraTreesClassifier(
        **CV_MODEL_PARAMS
    )

    log(
        f"[Fold {fold_index}/{N_SPLITS}] "
        f"model egitiliyor..."
    )

    fold_model.fit(
        X_train_fold_clean,
        y_train_fold
    )

    log(
        f"[Fold {fold_index}/{N_SPLITS}] "
        f"tahmin yapiliyor..."
    )

    fold_pred = fold_model.predict(
        X_test_fold_clean
    )

    fold_proba = fold_model.predict_proba(
        X_test_fold_clean
    )

    oof_pred_encoded[test_idx] = fold_pred

    oof_confidence[test_idx] = (
        fold_proba.max(axis=1)
    )

    fold_accuracy = accuracy_score(
        y_test_fold,
        fold_pred
    )

    fold_balanced_accuracy = (
        balanced_accuracy_score(
            y_test_fold,
            fold_pred
        )
    )

    fold_macro_f1 = f1_score(
        y_test_fold,
        fold_pred,
        average="macro",
        zero_division=0
    )

    fold_weighted_f1 = f1_score(
        y_test_fold,
        fold_pred,
        average="weighted",
        zero_division=0
    )

    fold_metrics_records.append({
        "fold": fold_index,
        "train_size": len(train_idx),
        "test_size": len(test_idx),
        "accuracy": fold_accuracy,
        "balanced_accuracy": fold_balanced_accuracy,
        "macro_f1": fold_macro_f1,
        "weighted_f1": fold_weighted_f1
    })

    log(
        f"Fold {fold_index}/{N_SPLITS} | "
        f"train={len(train_idx)} | "
        f"test={len(test_idx)} | "
        f"Acc={fold_accuracy:.4f} | "
        f"BalAcc={fold_balanced_accuracy:.4f} | "
        f"MacroF1={fold_macro_f1:.4f} | "
        f"WeightedF1={fold_weighted_f1:.4f}"
    )

    del (
        X_train_fold,
        X_test_fold,
        X_train_fold_clean,
        X_test_fold_clean,
        y_train_fold,
        y_test_fold,
        fold_model,
        fold_imputer,
        fold_pred,
        fold_proba
    )

    gc.collect()


assert (oof_pred_encoded != -1).all(), (
    "Bazi satirlar hicbir fold icinde test edilmedi."
)


fold_metrics_df = pd.DataFrame(
    fold_metrics_records
)

fold_metrics_df.to_csv(
    FOLD_METRICS_PATH,
    index=False
)


log("\nFold bazli metrikler:")
log(fold_metrics_df)

log("\nFold ortalama ve standart sapmalari:")

log(
    fold_metrics_df[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1"
        ]
    ].agg(["mean", "std"])
)


# ============================================================
# 21. OOF GENEL METRIKLER
# ============================================================

overall_accuracy = accuracy_score(
    y_encoded,
    oof_pred_encoded
)

overall_balanced_accuracy = (
    balanced_accuracy_score(
        y_encoded,
        oof_pred_encoded
    )
)

overall_macro_f1 = f1_score(
    y_encoded,
    oof_pred_encoded,
    average="macro",
    zero_division=0
)

overall_weighted_f1 = f1_score(
    y_encoded,
    oof_pred_encoded,
    average="weighted",
    zero_division=0
)


log("\n" + "=" * 80)
log("OOF GENEL MODEL SONUCLARI")
log("=" * 80)

log(
    "Accuracy:",
    round(overall_accuracy, 4)
)

log(
    "Balanced Accuracy:",
    round(overall_balanced_accuracy, 4)
)

log(
    "Macro F1:",
    round(overall_macro_f1, 4)
)

log(
    "Weighted F1:",
    round(overall_weighted_f1, 4)
)


report_text = classification_report(
    y_encoded,
    oof_pred_encoded,
    labels=np.arange(
        len(label_encoder.classes_)
    ),
    target_names=label_encoder.classes_,
    zero_division=0,
    digits=4
)

log("\nOOF Classification Report:\n")
log(report_text)


report_dict = classification_report(
    y_encoded,
    oof_pred_encoded,
    labels=np.arange(
        len(label_encoder.classes_)
    ),
    target_names=label_encoder.classes_,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(
    report_dict
).transpose()

report_df.to_csv(
    REPORT_PATH,
    index=True
)


# ============================================================
# 22. LABEL ISIMLERINE GERI DON
# ============================================================

oof_true_labels_all = (
    label_encoder.inverse_transform(
        y_encoded
    )
)

oof_pred_labels_all = (
    label_encoder.inverse_transform(
        oof_pred_encoded
    )
)


# ============================================================
# 23. INFILTRATION OZEL ANALIZI
# ============================================================

infiltration_mask = (
    oof_true_labels_all
    == "Infiltration"
)

if infiltration_mask.any():

    infiltration_correct = (
        oof_pred_labels_all[infiltration_mask]
        == "Infiltration"
    )

    log("\n" + "=" * 80)
    log("INFILTRATION OOF PERFORMANSI")
    log("=" * 80)

    log(
        "Toplam Infiltration:",
        int(infiltration_mask.sum())
    )

    log(
        "Dogru tahmin:",
        int(infiltration_correct.sum())
    )

    log(
        "OOF Recall:",
        round(
            float(infiltration_correct.mean()),
            4
        )
    )

    infiltration_by_dataset = (
        pd.DataFrame({
            "dataset": dataset_names_all[
                infiltration_mask
            ].to_numpy(),

            "source_file": source_files_all[
                infiltration_mask
            ].to_numpy(),

            "is_correct": infiltration_correct
        })
        .groupby(
            ["dataset", "source_file"]
        )["is_correct"]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "n_samples",
            "mean": "oof_recall"
        })
    )

    log(
        "\nDataset ve dosya bazinda "
        "Infiltration recall:"
    )

    log(infiltration_by_dataset)

    infiltration_by_dataset.to_csv(
        INFILTRATION_BY_SOURCE_PATH
    )

else:
    log("\nInfiltration sinifi bulunamadi.")


# ============================================================
# 24. WEBATTACKCANDIDATE OZEL ANALIZI
# ============================================================

web_candidate_true_mask = (
    oof_true_labels_all
    == WEB_ATTACK_CANDIDATE_LABEL
)

web_candidate_pred_mask = (
    oof_pred_labels_all
    == WEB_ATTACK_CANDIDATE_LABEL
)


web_candidate_metrics = {
    "precision": None,
    "recall": None,
    "f1": None,
    "true_samples": 0,
    "predicted_samples": 0
}


if web_candidate_true_mask.any():

    web_candidate_y_true_binary = (
        oof_true_labels_all
        == WEB_ATTACK_CANDIDATE_LABEL
    ).astype(int)

    web_candidate_y_pred_binary = (
        oof_pred_labels_all
        == WEB_ATTACK_CANDIDATE_LABEL
    ).astype(int)

    web_precision = precision_score(
        web_candidate_y_true_binary,
        web_candidate_y_pred_binary,
        zero_division=0
    )

    web_recall = recall_score(
        web_candidate_y_true_binary,
        web_candidate_y_pred_binary,
        zero_division=0
    )

    web_f1 = f1_score(
        web_candidate_y_true_binary,
        web_candidate_y_pred_binary,
        zero_division=0
    )

    correctly_routed = (
        oof_pred_labels_all[
            web_candidate_true_mask
        ]
        == WEB_ATTACK_CANDIDATE_LABEL
    )

    web_candidate_metrics = {
        "precision": float(web_precision),
        "recall": float(web_recall),
        "f1": float(web_f1),
        "true_samples": int(
            web_candidate_true_mask.sum()
        ),
        "predicted_samples": int(
            web_candidate_pred_mask.sum()
        )
    }

    log("\n" + "=" * 80)
    log("WEBATTACKCANDIDATE OOF PERFORMANSI")
    log("=" * 80)

    log(
        "Gercek WebAttackCandidate:",
        int(web_candidate_true_mask.sum())
    )

    log(
        "WebAttackCandidate olarak tahmin edilen:",
        int(web_candidate_pred_mask.sum())
    )

    log(
        "Payload katmanina dogru yonlendirilen:",
        int(correctly_routed.sum())
    )

    log(
        "Precision:",
        round(web_precision, 4)
    )

    log(
        "Recall:",
        round(web_recall, 4)
    )

    log(
        "F1:",
        round(web_f1, 4)
    )

    web_candidate_analysis_df = pd.DataFrame({
        "flow_id": flow_ids_all[
            web_candidate_true_mask
        ].to_numpy(),

        "dataset": dataset_names_all[
            web_candidate_true_mask
        ].to_numpy(),

        "source_file": source_files_all[
            web_candidate_true_mask
        ].to_numpy(),

        "original_web_label": original_labels_all[
            web_candidate_true_mask
        ].to_numpy(),

        "flow_model_true_label": oof_true_labels_all[
            web_candidate_true_mask
        ],

        "predicted_attack": oof_pred_labels_all[
            web_candidate_true_mask
        ],

        "confidence": oof_confidence[
            web_candidate_true_mask
        ],

        "correctly_routed_to_payload": correctly_routed
    })

    web_candidate_analysis_df.to_csv(
        WEB_CANDIDATE_ANALYSIS_PATH,
        index=False
    )

    web_original_label_performance = (
        web_candidate_analysis_df
        .groupby("original_web_label")[
            "correctly_routed_to_payload"
        ]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "n_samples",
            "mean": "routing_recall"
        })
        .sort_values(
            "n_samples",
            ascending=False
        )
    )

    log(
        "\nOrijinal web label bazinda "
        "payload routing basarisi:"
    )

    log(web_original_label_performance)

    web_original_label_performance.to_csv(
        WEB_CANDIDATE_BY_ORIGINAL_LABEL_PATH
    )

else:
    log(
        "\nWebAttackCandidate sinifi "
        "egitim verisinde bulunamadi."
    )


# ============================================================
# 25. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_encoded,
    oof_pred_encoded,
    labels=np.arange(
        len(label_encoder.classes_)
    )
)


plt.figure(
    figsize=(
        max(
            12,
            len(label_encoder.classes_) * 1.1
        ),
        max(
            10,
            len(label_encoder.classes_) * 0.9
        )
    )
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title(
    "Attack-Type Classification - "
    "OOF Confusion Matrix"
)

plt.xlabel("Tahmin Edilen Saldiri Turu")
plt.ylabel("Gercek Saldiri Turu")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.yticks(rotation=0)

plt.tight_layout()

plt.savefig(
    CONFUSION_MATRIX_PATH,
    dpi=150,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 26. OOF TAHMINLERINI KAYDET
# ============================================================

route_to_payload_layer_oof = (
    oof_pred_labels_all
    == WEB_ATTACK_CANDIDATE_LABEL
)

prediction_df = pd.DataFrame({
    "flow_id": flow_ids_all.to_numpy(),

    "source_file": source_files_all.to_numpy(),

    "dataset": dataset_names_all.to_numpy(),

    "original_label": original_labels_all.to_numpy(),

    "true_attack": oof_true_labels_all,

    "predicted_attack": oof_pred_labels_all,

    "confidence": oof_confidence,

    "is_correct": (
        oof_true_labels_all
        == oof_pred_labels_all
    ),

    "payload_analysis_expected": (
        oof_true_labels_all
        == WEB_ATTACK_CANDIDATE_LABEL
    ),

    "route_to_payload_layer": (
        route_to_payload_layer_oof
    ),

    "next_layer": np.where(
        route_to_payload_layer_oof,
        "http_payload_analyzer",
        "mitre_mapping"
    )
})

prediction_df.to_csv(
    PREDICTIONS_PATH,
    index=False
)


# ============================================================
# 27. DATASET OZETI
# ============================================================

dataset_summary = (
    attack_df
    .groupby(
        [
            "_dataset_name",
            "_source_file",
            "Label",
            "_original_label"
        ]
    )
    .size()
    .reset_index(
        name="sample_count"
    )
    .sort_values(
        [
            "_dataset_name",
            "Label",
            "sample_count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

dataset_summary.to_csv(
    DATASET_SUMMARY_PATH,
    index=False
)

del attack_df
gc.collect()


# ============================================================
# 28. NIHAI PRODUCTION MODELINI TUM VERIYLE EGIT
# ============================================================

log(
    "\nNihai production model egitiliyor. "
    f"n_estimators={FINAL_N_ESTIMATORS}"
)

final_imputer = SimpleImputer(
    strategy="median"
)

X_all_clean = (
    final_imputer
    .fit_transform(X_all)
    .astype("float32")
)

final_model = ExtraTreesClassifier(
    **FINAL_MODEL_PARAMS
)

final_model.fit(
    X_all_clean,
    y_encoded
)

log("Nihai model egitimi tamamlandi.")


# ============================================================
# 29. FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": final_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

importance_df.to_csv(
    FEATURE_IMPORTANCE_PATH,
    index=False
)

log("\nEn onemli 30 feature:")

log(
    importance_df
    .head(30)
    .to_string(index=False)
)


# ============================================================
# 30. MODEL BUNDLE
# ============================================================

model_bundle = {
    "model": final_model,

    "imputer": final_imputer,

    "label_encoder": label_encoder,

    "features": FEATURES,

    "requested_features": REQUESTED_FEATURES,

    "column_mapping": COLUMN_MAPPING_2018_TO_2017,

    "random_state": RANDOM_STATE,

    "task": "hierarchical_attack_type_routing",

    "benign_expected": False,

    "web_attack_candidate_label": (
        WEB_ATTACK_CANDIDATE_LABEL
    ),

    "web_candidate_confidence_threshold": (
        WEB_CANDIDATE_CONFIDENCE_THRESHOLD
    ),

    "architecture": {
        "layer_0": (
            "Morpheus malicious traffic detection"
        ),
        "layer_1": (
            "Flow-based attack family classification "
            "(WebAttackCandidate routing sinifi dahil; "
            "Heartbleed degerlendirme disi birakilmistir; "
            "SQLi/XSS/BruteForce icin CSE-CIC-IDS2018 ek "
            "kaynaklari destekleniyor)"
        ),
        "layer_2": (
            "HTTP payload web attack classification "
            "(CSIC2010 tabanli - bkz. "
            "payload_classifier_csic2010.py)"
        ),
        "layer_3": (
            "MITRE mapping and agent response"
        )
    },

    "web_attack_policy": {
        "flow_output": WEB_ATTACK_CANDIDATE_LABEL,
        "flow_model_determines_sqli_or_xss": False,
        "requires_http_payload_analysis": True,
        "possible_payload_outputs": [
            "SQL Injection",
            "XSS",
            "Web Brute Force"
        ]
    },

    "rare_class_policy": {
        "rare_class_warning_threshold": (
            RARE_CLASS_WARNING_THRESHOLD
        ),
        "min_class_samples_for_training": (
            MIN_CLASS_SAMPLES
        ),
        "excluded_classes": {
            "Heartbleed": (
                "Dogal olarak ~11 kayitla anlamli CV "
                "degerlendirmesi yapilamadigi ve "
                "StratifiedKFold fold sayisini tum veri "
                "seti icin dusurdugu icin kaynaginda "
                "(CSV okuma asamasinda) tamamen cikarildi."
            )
        },
        "augmented_classes": {
            "WebAttackCandidate_SQLi": (
                "CICIDS2017'deki ~21 SQLi ornegi CSE-CIC-"
                "IDS2018'in ayri SQL Injection.csv / "
                "Brute Force -Web.csv / Brute Force -XSS.csv "
                "dosyalari (mevcutsa) ile zenginlestirildi."
            )
        }
    },

    "cv_strategy": "StratifiedKFold",
    "cv_n_splits": N_SPLITS,
    "cv_n_splits_limited_by_class": smallest_class_name,

    "metrics_oof": {
        "accuracy": float(overall_accuracy),
        "balanced_accuracy": float(overall_balanced_accuracy),
        "macro_f1": float(overall_macro_f1),
        "weighted_f1": float(overall_weighted_f1),
        "web_attack_candidate": web_candidate_metrics
    },

    "metrics_per_fold": (
        fold_metrics_df.to_dict(orient="records")
    )
}


joblib.dump(
    model_bundle,
    MODEL_PATH
)

log("\nModel paketi kaydedildi:", MODEL_PATH)


# ============================================================
# 31. MODELI YUKLEME TESTI
# ============================================================

loaded_bundle = joblib.load(MODEL_PATH)

loaded_model = loaded_bundle["model"]
loaded_imputer = loaded_bundle["imputer"]
loaded_encoder = loaded_bundle["label_encoder"]
loaded_features = loaded_bundle["features"]


# Rastgele TEK bir ornek yerine, HER SINIFTAN birer ornek
# alarak daha temsili bir saglik testi yapiyoruz. Boylece
# "tek kayit yanlis tahmin edildi" gibi yaniltici bir izlenim
# olusmaz.
log("\nModel yukleme testi (her siniftan bir ornek):")

for class_name in label_encoder.classes_:

    class_mask = (y_all == class_name).to_numpy()

    if not class_mask.any():
        continue

    sample_position = np.where(class_mask)[0][0]

    sample_input = X_all[loaded_features].iloc[
        [sample_position]
    ]

    sample_clean = loaded_imputer.transform(sample_input)

    sample_pred_encoded = loaded_model.predict(sample_clean)

    sample_pred = loaded_encoder.inverse_transform(
        sample_pred_encoded
    )[0]

    sample_probability = loaded_model.predict_proba(
        sample_clean
    )[0]

    sample_confidence = float(sample_probability.max())

    match_symbol = "OK" if sample_pred == class_name else "X"

    log(
        f"  [{match_symbol}] gercek={class_name} | "
        f"tahmin={sample_pred} | "
        f"confidence={round(sample_confidence, 4)}"
    )


# ============================================================
# 32. MORPHEUS SONRASI FLOW TAHMIN FONKSIYONU
# ============================================================

def predict_attack_type(
    flow_dataframe,
    model_bundle,
    web_candidate_confidence_threshold=None
):
    """
    Morpheus tarafindan tehlikeli olarak isaretlenen flow
    kayitlarinin saldiri ailesini tahmin eder.

    Kesin flow imzasina sahip saldirilar (DDoS, PortScan,
    Botnet, Infiltration, FTP/SSH-Patator vb.) dogrudan
    nihai sinif olarak doner.

    WebAttackCandidate sonucu ise kesin SQLi veya XSS
    sonucu DEGILDIR; ilgili flow'un HTTP payload analiz
    katmanina (payload_classifier_csic2010.py -
    classify_web_payload) yonlendirilmesi gerektigini
    belirtir. Bu katman, SQLi/XSS bulamazsa eleme
    yontemiyle "Web Brute Force" doner.

    NOT: Heartbleed bu modelde bir sinif olarak yer almaz;
    egitim verisinden kaynaginda cikarilmistir.
    """

    if not isinstance(flow_dataframe, pd.DataFrame):
        raise TypeError(
            "flow_dataframe bir pandas DataFrame olmalidir."
        )

    if flow_dataframe.empty:
        return pd.DataFrame(
            columns=[
                "flow_id",
                "predicted_attack",
                "confidence",
                "is_web_attack_candidate",
                "route_to_payload_layer",
                "next_layer",
                "decision_status",
                "final_attack_type_confirmed"
            ]
        )

    prediction_input = flow_dataframe.copy()

    prediction_input.columns = (
        prediction_input.columns
        .astype(str)
        .str.strip()
    )

    applicable_mapping = {
        old_name: new_name
        for old_name, new_name
        in model_bundle["column_mapping"].items()
        if (
            old_name in prediction_input.columns
            and new_name not in prediction_input.columns
        )
    }

    prediction_input.rename(
        columns=applicable_mapping,
        inplace=True
    )

    expected_features = model_bundle["features"]

    missing_features = [
        feature
        for feature in expected_features
        if feature not in prediction_input.columns
    ]

    if missing_features:
        raise ValueError(
            "Tahmin girdisinde modelin bekledigi "
            "feature'lar eksik:\n"
            + "\n".join(missing_features)
        )

    if "flow_id" in prediction_input.columns:

        input_flow_ids = (
            prediction_input["flow_id"]
            .astype(str)
            .reset_index(drop=True)
        )

    else:

        input_flow_ids = pd.Series([
            f"runtime-flow-{index}"
            for index in range(
                len(prediction_input)
            )
        ])

    prediction_X = prediction_input[
        expected_features
    ].copy()

    for column in expected_features:

        prediction_X[column] = pd.to_numeric(
            prediction_X[column],
            errors="coerce"
        ).astype("float32")

    prediction_X.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    prediction_X_clean = model_bundle[
        "imputer"
    ].transform(
        prediction_X
    ).astype("float32")

    encoded_predictions = model_bundle[
        "model"
    ].predict(
        prediction_X_clean
    )

    probabilities = model_bundle[
        "model"
    ].predict_proba(
        prediction_X_clean
    )

    attack_predictions = model_bundle[
        "label_encoder"
    ].inverse_transform(
        encoded_predictions
    )

    confidence_scores = probabilities.max(
        axis=1
    )

    web_candidate_label = model_bundle.get(
        "web_attack_candidate_label",
        WEB_ATTACK_CANDIDATE_LABEL
    )

    if web_candidate_confidence_threshold is None:

        web_candidate_confidence_threshold = (
            model_bundle.get(
                "web_candidate_confidence_threshold",
                WEB_CANDIDATE_CONFIDENCE_THRESHOLD
            )
        )

    is_web_attack_candidate = (
        attack_predictions
        == web_candidate_label
    )

    high_confidence_web_candidate = (
        is_web_attack_candidate
        & (
            confidence_scores
            >= web_candidate_confidence_threshold
        )
    )

    low_confidence_web_candidate = (
        is_web_attack_candidate
        & (
            confidence_scores
            < web_candidate_confidence_threshold
        )
    )

    route_to_payload_layer = (
        high_confidence_web_candidate
        | low_confidence_web_candidate
    )

    next_layer = np.where(
        route_to_payload_layer,
        "http_payload_analyzer",
        "mitre_mapping"
    )

    decision_status = np.where(
        high_confidence_web_candidate,
        "WEB_ATTACK_REQUIRES_PAYLOAD_ANALYSIS",
        np.where(
            low_confidence_web_candidate,
            "LOW_CONFIDENCE_WEB_CANDIDATE_REQUIRES_REVIEW",
            "FLOW_ATTACK_CLASSIFICATION_COMPLETE"
        )
    )

    final_attack_type_confirmed = (
        ~is_web_attack_candidate
    )

    return pd.DataFrame({
        "flow_id": input_flow_ids.to_numpy(),
        "predicted_attack": attack_predictions,
        "confidence": confidence_scores,
        "is_web_attack_candidate": is_web_attack_candidate,
        "route_to_payload_layer": route_to_payload_layer,
        "next_layer": next_layer,
        "decision_status": decision_status,
        "final_attack_type_confirmed": (
            final_attack_type_confirmed
        )
    })


# ============================================================
# 33. FLOW'LARI SONRAKI KATMANLARA AYIR
# ============================================================

def split_predictions_by_next_layer(
    original_flow_dataframe,
    prediction_dataframe
):
    """
    Tahmin edilen flow kayitlarini iki gruba ayirir:

    payload_candidates:
        HTTP payload analizine gidecek WebAttackCandidate
        kayitlari (Layer 2 - payload_classifier_csic2010.py
        tarafindan islenecek).

    flow_classification_complete:
        Flow modelinde saldiri ailesi kesinlesmis diger
        kayitlar.
    """

    if (
        len(original_flow_dataframe)
        != len(prediction_dataframe)
    ):
        raise ValueError(
            "Flow dataframe ile prediction dataframe "
            "satir sayilari esit degil."
        )

    original_flows = (
        original_flow_dataframe
        .reset_index(drop=True)
        .copy()
    )

    predictions = (
        prediction_dataframe
        .reset_index(drop=True)
        .copy()
    )

    combined = pd.concat(
        [
            original_flows,
            predictions.add_prefix("routing_")
        ],
        axis=1
    )

    payload_mask = predictions[
        "route_to_payload_layer"
    ].astype(bool)

    payload_candidates = combined[payload_mask].copy()

    flow_classification_complete = combined[
        ~payload_mask
    ].copy()

    return {
        "payload_candidates": payload_candidates,
        "flow_classification_complete": (
            flow_classification_complete
        )
    }


# ============================================================
# 34. TAMAMLANDI
# ============================================================

log("\n" + "=" * 80)
log("PIPELINE TAMAMLANDI (LAYER 1 - FLOW CLASSIFIER)")
log("=" * 80)

log("Model:", MODEL_PATH)
log("OOF report:", REPORT_PATH)
log("OOF predictions:", PREDICTIONS_PATH)
log("Confusion matrix:", CONFUSION_MATRIX_PATH)
log("Feature importance:", FEATURE_IMPORTANCE_PATH)

log(
    "WebAttackCandidate analizi:",
    WEB_CANDIDATE_ANALYSIS_PATH
)

log(
    "\nNOT: Heartbleed bu pipeline'da degerlendirme disi "
    "birakilmistir."
)

log(
    "NOT: SQLi/XSS/BruteForce icin CSE-CIC-IDS2018 ek "
    "kaynaklari (varsa) kullanilmistir - bkz. bolum 8B."
)

log("\nMorpheus sonrasi kullanim:")

log(
    "flow_predictions = predict_attack_type("
    "morpheus_attack_flows, loaded_bundle)"
)



Dosya okunuyor: data/Monday-WorkingHours.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (529918, 79)

BENIGN filtresi oncesi siniflar:
Label
BENIGN    529918
Name: count, dtype: int64
Bu dosyada saldiri kaydi bulunamadi.

Dosya okunuyor: data/Tuesday-WorkingHours.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (445909, 79)

BENIGN filtresi oncesi siniflar:
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Kalan saldiri kayitlari:
Label
FTP-Patator    7938
SSH-Patator    5897
Name: count, dtype: int64

Dosya okunuyor: data/Wednesday-workingHours.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (692703, 79)

Heartbleed kaydi bulundu ve degerlendirme disi birakildi. Cikarilan kayit sayisi: 11

BENIGN filtresi oncesi siniflar:
Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Name: count, dtype: int64

Kalan saldiri kayitlari:
Label
DoS Hulk            231073


In [10]:
# ============================================================
# ATTACK-TYPE CLASSIFIER (FLOW / LAYER-1)
# HİBRİT EĞİTİM VERSİYONU (2017 + 2018 + IDS2025 BİRLEŞTİRİLMİŞ)
# THERMAL-THROTTLING ÖNLEMLİ (DOWNSAMPLED)
# ============================================================

import os
import re
import gc
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

def log(*args):
    print(*args, flush=True)

# ============================================================
# 1. GENEL AYARLAR
# ============================================================
RANDOM_STATE = 42
N_SPLITS_MAX = 5
N_SPLITS_MIN = 2

# Ağaç sayıları işlemciyi korumak için optimize edildi
CV_N_ESTIMATORS = 50
FINAL_N_ESTIMATORS = 200
N_JOBS = 4

MIN_CLASS_SAMPLES = 10
# Her sınıftan alınacak maksimum örnek sayısı (Aşırı yükü engeller)
MAX_SAMPLES_PER_CLASS = 30000

WEB_ATTACK_CANDIDATE_LABEL = "WebAttackCandidate"
WEB_CANDIDATE_CONFIDENCE_THRESHOLD = 0.50
RARE_CLASS_WARNING_THRESHOLD = 20

# ============================================================
# 2. DOSYA VE KLASOR AYARLARI
# ============================================================
DATA_DIR = "data"
MODEL_DIR = "models"
OUTPUT_DIR = "outputs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

CICIDS_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

CSE_CIC_IDS_2018_INFILTRATION_FILE = os.path.join(DATA_DIR, "Wednesday-28-02-2018.csv")

CSE_CIC_IDS_2018_WEBATTACK_FILES = [
    os.path.join(DATA_DIR, "SQL Injection.csv"),
    os.path.join(DATA_DIR, "Brute Force -Web.csv"),
    os.path.join(DATA_DIR, "Brute Force -XSS.csv"),
]

CSE_CIC_IDS_2018_EXTRA_FILE = os.path.join(DATA_DIR, "2018", "cic.csv")

# YENİ 2025 VERİ SETİ
IDS2025_FILE = "data/IDS2025.xlsx"

MODEL_PATH = os.path.join(MODEL_DIR, "attack_classifier.joblib")
REPORT_PATH = os.path.join(OUTPUT_DIR, "attack_classification_report_oof.csv")
FOLD_METRICS_PATH = os.path.join(OUTPUT_DIR, "attack_fold_metrics.csv")
CONFUSION_MATRIX_PATH = os.path.join(OUTPUT_DIR, "attack_confusion_matrix_oof.png")
FEATURE_IMPORTANCE_PATH = os.path.join(OUTPUT_DIR, "attack_feature_importance.csv")
DATASET_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "attack_dataset_summary.csv")
PREDICTIONS_PATH = os.path.join(OUTPUT_DIR, "attack_oof_predictions.csv")

# ============================================================
# 3. ORTAK FEATURE SEMASI
# ============================================================
REQUESTED_FEATURES = [
    "Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets", "Fwd Packet Length Max",
    "Fwd Packet Length Min", "Fwd Packet Length Mean", "Fwd Packet Length Std",
    "Bwd Packet Length Max", "Bwd Packet Length Min", "Bwd Packet Length Mean",
    "Bwd Packet Length Std", "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean",
    "Flow IAT Std", "Flow IAT Max", "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean",
    "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean",
    "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags",
    "Fwd URG Flags", "Bwd URG Flags", "Fwd Header Length", "Bwd Header Length",
    "Fwd Packets/s", "Bwd Packets/s", "Min Packet Length", "Max Packet Length",
    "Packet Length Mean", "Packet Length Std", "Packet Length Variance",
    "FIN Flag Count", "SYN Flag Count", "RST Flag Count", "PSH Flag Count",
    "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
    "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
    "Fwd Header Length.1", "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate", "Subflow Fwd Packets", "Subflow Fwd Bytes",
    "Subflow Bwd Packets", "Subflow Bwd Bytes", "Init_Win_bytes_forward",
    "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward",
    "Active Mean", "Active Std", "Active Max", "Active Min", "Idle Mean",
    "Idle Std", "Idle Max", "Idle Min"
]

# ============================================================
# 4. CSE-CIC-IDS2018 & IDS2025 KOLONLARINI CICIDS2017 SEMASINA ÇEVİR
# ============================================================
COLUMN_MAPPING_2018_TO_2017 = {
    "Dst Port": "Destination Port", "Tot Fwd Pkts": "Total Fwd Packets",
    "Tot Bwd Pkts": "Total Backward Packets", "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets", "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min", "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std", "Bwd Pkt Len Max": "Bwd Packet Length Max",
    "Bwd Pkt Len Min": "Bwd Packet Length Min", "Bwd Pkt Len Mean": "Bwd Packet Length Mean",
    "Bwd Pkt Len Std": "Bwd Packet Length Std", "Flow Byts/s": "Flow Bytes/s",
    "Flow Pkts/s": "Flow Packets/s", "Fwd IAT Tot": "Fwd IAT Total",
    "Bwd IAT Tot": "Bwd IAT Total", "Fwd Header Len": "Fwd Header Length",
    "Bwd Header Len": "Bwd Header Length", "Fwd Pkts/s": "Fwd Packets/s",
    "Bwd Pkts/s": "Bwd Packets/s", "Pkt Len Min": "Min Packet Length",
    "Pkt Len Max": "Max Packet Length", "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std", "Pkt Len Var": "Packet Length Variance",
    "FIN Flag Cnt": "FIN Flag Count", "SYN Flag Cnt": "SYN Flag Count",
    "RST Flag Cnt": "RST Flag Count", "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count", "URG Flag Cnt": "URG Flag Count",
    "CWE Flag Count": "CWE Flag Count", "ECE Flag Cnt": "ECE Flag Count",
    "Pkt Size Avg": "Average Packet Size", "Fwd Seg Size Avg": "Avg Fwd Segment Size",
    "Bwd Seg Size Avg": "Avg Bwd Segment Size", "Fwd Byts/b Avg": "Fwd Avg Bytes/Bulk",
    "Fwd Pkts/b Avg": "Fwd Avg Packets/Bulk", "Fwd Blk Rate Avg": "Fwd Avg Bulk Rate",
    "Bwd Byts/b Avg": "Bwd Avg Bytes/Bulk", "Bwd Pkts/b Avg": "Bwd Avg Packets/Bulk",
    "Bwd Blk Rate Avg": "Bwd Avg Bulk Rate", "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Fwd Byts": "Subflow Fwd Bytes", "Subflow Bwd Pkts": "Subflow Bwd Packets",
    "Subflow Bwd Byts": "Subflow Bwd Bytes", "Init Fwd Win Byts": "Init_Win_bytes_forward",
    "Init Bwd Win Byts": "Init_Win_bytes_backward", "Fwd Act Data Pkts": "act_data_pkt_fwd",
    "Fwd Seg Size Min": "min_seg_size_forward",
    # IDS2025 Excel dosyası özel eşlemeleri
    "Flow Bytess": "Flow Bytes/s", "Flow Packetss": "Flow Packets/s"
}

# ============================================================
# 5. LABEL NORMALIZASYONU (MACRO-CLASS)
# ============================================================
def normalize_label(label):
    label = str(label).strip()
    label = re.sub(r"\s+", " ", label)
    label = label.replace("ï¿½", "-").replace("–", "-").replace("—", "-")
    lower_label = label.casefold()

    if lower_label == "label": return "HEADER_ROW"
    if lower_label in {"benign", "normal"}: return "BENIGN"
    if lower_label in {"infilteration", "infiltration"}: return "Infiltration"
    if lower_label in {"bot", "botnet", "botnet ares"}: return "Botnet"
    if lower_label in {"portscan", "port scan"}: return "PortScan"

    # WEB ATTACK CANDIDATE
    if "web attack" in lower_label: return WEB_ATTACK_CANDIDATE_LABEL
    explicit_web_labels = {"brute force -web", "brute force - web", "brute force -xss", "brute force - xss", "sql injection", "sql-injection", "sqli", "xss", "cross site scripting", "cross-site scripting"}
    if lower_label in explicit_web_labels: return WEB_ATTACK_CANDIDATE_LABEL
    compact_label = re.sub(r"[\s_\-]+", "", lower_label)
    compact_web_labels = {"webattackbruteforce", "webattackxss", "webattacksqlinjection", "bruteforceweb", "bruteforcexss", "sqlinjection", "sqli", "xss", "crosssitescripting"}
    if compact_label in compact_web_labels: return WEB_ATTACK_CANDIDATE_LABEL

    # BRUTE FORCE & PATATOR (FTP / SSH / Genel)
    if "ftp" in lower_label and ("patator" in lower_label or "brute" in lower_label):
        return "FTP-BruteForce"
    if "ssh" in lower_label and ("patator" in lower_label or "brute" in lower_label):
        return "SSH-BruteForce"
    if lower_label == "brute force": # IDS2025 genel brute force etiketi
        return "FTP-BruteForce"

    # DDOS / DOS
    if lower_label in {"ddos", "dos/ddos"}: return "DDoS"
    if "ddos" in lower_label:
        if "hoic" in lower_label: return "DDoS-HOIC"
        if "loic" in lower_label and "udp" in lower_label: return "DDoS-LOIC-UDP"
        if "loic" in lower_label: return "DDoS-LOIC-HTTP"
        return "DDoS"

    if "slowhttptest" in lower_label: return "DoS Slowhttptest"
    if "slowloris" in lower_label: return "DoS slowloris"
    if "goldeneye" in lower_label: return "DoS GoldenEye"
    if "hulk" in lower_label: return "DoS Hulk"

    return label

# ============================================================
# 6. CSV OKUMA VE TEMIZLEME
# ============================================================
def load_and_prepare_csv(file_path, dataset_name):
    log("\n" + "=" * 80)
    log("Dosya okunuyor:", file_path)
    log("Dataset:", dataset_name)

    if not os.path.exists(file_path):
        log("UYARI: Dosya bulunamadi, atlandi.")
        return None

    current_df = pd.read_csv(file_path, low_memory=False)
    log("Ham boyut:", current_df.shape)
    current_df.columns = current_df.columns.astype(str).str.strip()

    if "Label" not in current_df.columns:
        log("UYARI: Label kolonu bulunamadi, atlandi.")
        return None

    applicable_mapping = {old: new for old, new in COLUMN_MAPPING_2018_TO_2017.items() if old in current_df.columns and new not in current_df.columns}
    if applicable_mapping:
        current_df.rename(columns=applicable_mapping, inplace=True)

    if "Fwd Header Length.1" not in current_df.columns and "Fwd Header Length" in current_df.columns:
        current_df["Fwd Header Length.1"] = current_df["Fwd Header Length"]

    current_df["_original_label"] = current_df["Label"].astype(str).str.strip()
    heartbleed_mask = current_df["_original_label"].str.contains("heartbleed", case=False, na=False)

    if heartbleed_mask.any():
        current_df = current_df[~heartbleed_mask].copy()

    current_df["Label"] = current_df["Label"].apply(normalize_label)
    current_df = current_df[current_df["Label"] != "HEADER_ROW"].copy()
    current_df = current_df[current_df["Label"] != "BENIGN"].copy()

    if current_df.empty:
        log("Bu dosyada saldiri kaydi bulunamadi.")
        return None

    current_df["_source_file"] = Path(file_path).name
    current_df["_dataset_name"] = dataset_name
    current_df["_requires_payload_analysis"] = (current_df["Label"] == WEB_ATTACK_CANDIDATE_LABEL)

    if "flow_id" not in current_df.columns:
        current_df["flow_id"] = [f"{dataset_name}:{Path(file_path).name}:{index}" for index in current_df.index]
    else:
        missing_flow_id_mask = current_df["flow_id"].isna() | (current_df["flow_id"].astype(str).str.strip() == "")
        generated_ids = np.array([f"{dataset_name}:{Path(file_path).name}:{index}" for index in current_df.index], dtype=object)
        current_df.loc[missing_flow_id_mask, "flow_id"] = generated_ids[missing_flow_id_mask.to_numpy()]

    log("\nKalan saldiri kayitlari:")
    log(current_df["Label"].value_counts())
    return current_df


# ============================================================
# 6.5. EXCEL OKUMA VE TEMIZLEME (IDS2025 İÇİN)
# ============================================================
def load_and_prepare_excel(file_path, dataset_name):
    log("\n" + "=" * 80)
    log("Excel dosyasi okunuyor:", file_path)
    log("Dataset:", dataset_name)

    if not os.path.exists(file_path):
        log("UYARI: Dosya bulunamadi, atlandi.")
        return None

    current_df = pd.read_excel(file_path)
    log("Ham boyut:", current_df.shape)
    current_df.columns = current_df.columns.astype(str).str.strip()

    if "newLabel" in current_df.columns:
        current_df.rename(columns={"newLabel": "Label"}, inplace=True)
    elif "Label" not in current_df.columns:
        log("UYARI: Label kolonu bulunamadi, atlandi.")
        return None

    applicable_mapping = {old: new for old, new in COLUMN_MAPPING_2018_TO_2017.items() if old in current_df.columns and new not in current_df.columns}
    if applicable_mapping:
        current_df.rename(columns=applicable_mapping, inplace=True)

    if "Fwd Header Length.1" not in current_df.columns and "Fwd Header Length" in current_df.columns:
        current_df["Fwd Header Length.1"] = current_df["Fwd Header Length"]

    current_df["_original_label"] = current_df["Label"].astype(str).str.strip()
    current_df["Label"] = current_df["Label"].apply(normalize_label)

    current_df = current_df[current_df["Label"] != "BENIGN"].copy()

    if current_df.empty:
        log("Bu dosyada saldiri kaydi bulunamadi.")
        return None

    current_df["_source_file"] = Path(file_path).name
    current_df["_dataset_name"] = dataset_name
    current_df["_requires_payload_analysis"] = (current_df["Label"] == WEB_ATTACK_CANDIDATE_LABEL)
    current_df["flow_id"] = [f"{dataset_name}:{Path(file_path).name}:{index}" for index in current_df.index]

    log("\nKalan saldiri kayitlari:")
    log(current_df["Label"].value_counts())
    return current_df


# ============================================================
# 7 - 8. VERİ SETLERİNİ YÜKLE
# ============================================================
attack_frames = []

# CICIDS2017
for file_name in CICIDS_FILES:
    file_path = os.path.join(DATA_DIR, file_name)
    prepared_df = load_and_prepare_csv(file_path, "CICIDS2017")
    if prepared_df is not None:
        attack_frames.append(prepared_df)

# CSE-CIC-IDS2018 Infiltration
infiltration_2018_df = load_and_prepare_csv(CSE_CIC_IDS_2018_INFILTRATION_FILE, "CSE-CIC-IDS2018")
if infiltration_2018_df is not None:
    infiltration_2018_df = infiltration_2018_df[infiltration_2018_df["Label"] == "Infiltration"].copy()
    if not infiltration_2018_df.empty:
        attack_frames.append(infiltration_2018_df)

# CSE-CIC-IDS2018 Web Attacks
for webattack_file_path in CSE_CIC_IDS_2018_WEBATTACK_FILES:
    webattack_2018_df = load_and_prepare_csv(webattack_file_path, "CSE-CIC-IDS2018-WebAttacks")
    if webattack_2018_df is not None:
        webattack_2018_df = webattack_2018_df[webattack_2018_df["Label"] == WEB_ATTACK_CANDIDATE_LABEL].copy()
        if not webattack_2018_df.empty:
            attack_frames.append(webattack_2018_df)

# CSE-CIC-IDS2018 General
cic_2018_df = load_and_prepare_csv(CSE_CIC_IDS_2018_EXTRA_FILE, "CSE-CIC-IDS2018-General")
if cic_2018_df is not None:
    attack_frames.append(cic_2018_df)

# IDS2025 Excel Dosyası
ids2025_df = load_and_prepare_excel(IDS2025_FILE, "IDS2025")
if ids2025_df is not None:
    attack_frames.append(ids2025_df)

# ============================================================
# 9. BİRLEŞTİR VE DOWNSAMPLE YAP
# ============================================================
if not attack_frames:
    raise ValueError("Hicbir saldiri dataseti yuklenemedi.")

attack_df = pd.concat(attack_frames, ignore_index=True, sort=False)
del attack_frames
gc.collect()

log("\n" + "=" * 80)
log("BIRLESTIRILMIS SALDIRI DATASETI")
log("=" * 80)
log("Toplam saldiri satiri:", len(attack_df))

class_counts = attack_df["Label"].value_counts()
too_small_classes = class_counts[class_counts < MIN_CLASS_SAMPLES].index.tolist()
if too_small_classes:
    attack_df = attack_df[~attack_df["Label"].isin(too_small_classes)].copy()

log(f"\nEğitim veri seti çok büyük, işlemciyi korumak için her sınıftan maks {MAX_SAMPLES_PER_CLASS} kayıt alınıyor...")
sampled_frames = []
for current_label, group in attack_df.groupby("Label"):
    sampled_group = group.sample(n=min(len(group), MAX_SAMPLES_PER_CLASS), random_state=RANDOM_STATE)
    sampled_frames.append(sampled_group)

attack_df = pd.concat(sampled_frames, ignore_index=True)
log("\nDownsampling sonrasi yeni sinif dagilimi:")
log(attack_df["Label"].value_counts())

# ============================================================
# 13 - 18. FEATURE SEÇİMİ VE ENCODING
# ============================================================
FEATURES = [feature for feature in REQUESTED_FEATURES if feature in attack_df.columns]
if not FEATURES:
    raise ValueError("Model egitimi icin kullanilabilir feature bulunamadi.")

X_all = attack_df[FEATURES].copy()
y_all = attack_df["Label"].copy()

for column in X_all.columns:
    X_all[column] = pd.to_numeric(X_all[column], errors="coerce").astype("float32")
X_all.replace([np.inf, -np.inf], np.nan, inplace=True)

X_all.dropna(axis=1, how="all", inplace=True)
constant_columns = [col for col in X_all.columns if X_all[col].nunique(dropna=True) <= 1]
X_all.drop(columns=constant_columns, inplace=True)
FEATURES = X_all.columns.tolist()

combined_for_duplicates = X_all.copy()
combined_for_duplicates["_label"] = y_all.to_numpy()
combined_for_duplicates = combined_for_duplicates.drop_duplicates(subset=FEATURES + ["_label"], keep="first").copy()
y_all = combined_for_duplicates.pop("_label")
X_all = combined_for_duplicates.reset_index(drop=True)
y_all = y_all.reset_index(drop=True)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_all)

# ============================================================
# 19 - 20. STRATIFIEDKFOLD VE DEĞERLENDİRME
# ============================================================
smallest_class_count = int(pd.Series(y_encoded).value_counts().min())
N_SPLITS = min(N_SPLITS_MAX, smallest_class_count)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
CV_MODEL_PARAMS = {"n_estimators": CV_N_ESTIMATORS, "max_features": "sqrt", "min_samples_leaf": 2, "class_weight": "balanced", "random_state": RANDOM_STATE, "n_jobs": N_JOBS}
FINAL_MODEL_PARAMS = {"n_estimators": FINAL_N_ESTIMATORS, "max_features": "sqrt", "min_samples_leaf": 2, "class_weight": "balanced", "random_state": RANDOM_STATE, "n_jobs": N_JOBS}

oof_pred_encoded = np.full(shape=len(X_all), fill_value=-1, dtype=int)

log("\n" + "=" * 80)
log(f"STRATIFIEDKFOLD BASLIYOR ({N_SPLITS} fold)")
log("=" * 80)

for fold_index, (train_idx, test_idx) in enumerate(skf.split(X_all, y_encoded), start=1):
    X_train_fold, X_test_fold = X_all.iloc[train_idx], X_all.iloc[test_idx]
    y_train_fold, y_test_fold = y_encoded[train_idx], y_encoded[test_idx]

    fold_imputer = SimpleImputer(strategy="median")
    X_train_clean = fold_imputer.fit_transform(X_train_fold).astype("float32")
    X_test_clean = fold_imputer.transform(X_test_fold).astype("float32")

    fold_model = ExtraTreesClassifier(**CV_MODEL_PARAMS)
    fold_model.fit(X_train_clean, y_train_fold)

    oof_pred_encoded[test_idx] = fold_model.predict(X_test_clean)
    fold_acc = accuracy_score(y_test_fold, oof_pred_encoded[test_idx])
    log(f"Fold {fold_index}/{N_SPLITS} | Acc={fold_acc:.4f}")

    del X_train_fold, X_test_fold, X_train_clean, X_test_clean, fold_model, fold_imputer
    gc.collect()

log("\nOOF Accuracy:", round(accuracy_score(y_encoded, oof_pred_encoded), 4))

log("\n" + "=" * 80)
log("SINIF (PROTOKOL) BAZINDA BAŞARI RAPORU")
log("=" * 80)
log(classification_report(y_encoded, oof_pred_encoded, target_names=label_encoder.classes_, digits=4))

# ============================================================
# 28. NİHAİ MODELİ EĞİT VE KAYDET
# ============================================================
log("\nNihai production model egitiliyor...")
final_imputer = SimpleImputer(strategy="median")
X_all_clean = final_imputer.fit_transform(X_all).astype("float32")

final_model = ExtraTreesClassifier(**FINAL_MODEL_PARAMS)
final_model.fit(X_all_clean, y_encoded)

model_bundle = {
    "model": final_model,
    "imputer": final_imputer,
    "label_encoder": label_encoder,
    "features": FEATURES,
    "requested_features": REQUESTED_FEATURES,
    "column_mapping": COLUMN_MAPPING_2018_TO_2017,
    "web_attack_candidate_label": WEB_ATTACK_CANDIDATE_LABEL
}

joblib.dump(model_bundle, MODEL_PATH)
log("\nModel paketi kaydedildi:", MODEL_PATH)
log("PIPELINE TAMAMLANDI!")


Dosya okunuyor: data/Monday-WorkingHours.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (529918, 79)
Bu dosyada saldiri kaydi bulunamadi.

Dosya okunuyor: data/Tuesday-WorkingHours.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (445909, 79)

Kalan saldiri kayitlari:
Label
FTP-BruteForce    7938
SSH-BruteForce    5897
Name: count, dtype: int64

Dosya okunuyor: data/Wednesday-workingHours.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (692703, 79)

Kalan saldiri kayitlari:
Label
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Name: count, dtype: int64

Dosya okunuyor: data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (170366, 79)

Kalan saldiri kayitlari:
Label
WebAttackCandidate    2180
Name: count, dtype: int64

Dosya okunuyor: data/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Dataset: CICIDS2017
Ham boyut: (288602, 79)

Kalan saldiri kayitlari:
Label
Infiltration    36
Name: 

In [11]:
log("\n" + "=" * 80)
log("SINIF (PROTOKOL) BAZINDA BAŞARI RAPORU")
log("=" * 80)
report_text = classification_report(
    y_encoded,
    oof_pred_encoded,
    target_names=label_encoder.classes_,
    digits=4
)
log(report_text)


SINIF (PROTOKOL) BAZINDA BAŞARI RAPORU
                    precision    recall  f1-score   support

            Botnet     0.9987    1.0000    0.9993      3813
              DDoS     0.9998    0.9951    0.9974     29164
     DoS GoldenEye     0.9958    0.9988    0.9973     10282
          DoS Hulk     0.9951    0.9990    0.9971     22934
  DoS Slowhttptest     0.9931    0.9933    0.9932      5228
     DoS slowloris     0.9905    0.9942    0.9924      5374
    FTP-BruteForce     0.9355    0.9965    0.9650      2022
      Infiltration     0.9995    0.9982    0.9989     28260
          PortScan     0.9999    0.9995    0.9997     26572
    SSH-BruteForce     0.9996    0.9913    0.9954     15204
WebAttackCandidate     0.9858    0.9924    0.9891      5033

          accuracy                         0.9968    153886
         macro avg     0.9903    0.9962    0.9932    153886
      weighted avg     0.9969    0.9968    0.9968    153886



In [12]:
import os
import joblib
import pandas as pd
import numpy as np

# 1. Kaydedilen model paketini yükle
MODEL_PATH = "models/attack_classifier.joblib"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model dosyası bulunamadı: {MODEL_PATH}")

print("Model paketi yükleniyor...")
bundle = joblib.load(MODEL_PATH)
model = bundle["model"]
imputer = bundle["imputer"]
label_encoder = bundle["label_encoder"]
features = bundle["features"]
column_mapping = bundle["column_mapping"]
web_candidate_label = bundle["web_attack_candidate_label"]

# 2. Test edilecek dosyaların listesi (Hem 2017 hem 2025/2018 karışık)
TEST_FILES = [
    "data/Tuesday-WorkingHours.pcap_ISCX.csv",  # CICIDS2017 (FTP/SSH BruteForce)
    "data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv", # CICIDS2017 (PortScan)
    "data/IDS2025.xlsx"                              # IDS2025 (Modern Karışık)
]

dfs = []
for file_path in TEST_FILES:
    if not os.path.exists(file_path):
        print(f"Uyarı: {file_path} bulunamadı, atlanıyor.")
        continue

    print(f"Test dosyası yükleniyor: {file_path}")
    if file_path.endswith(".xlsx"):
        temp_df = pd.read_excel(file_path)
        if "newLabel" in temp_df.columns:
            temp_df.rename(columns={"newLabel": "Label"}, inplace=True)
    else:
        temp_df = pd.read_csv(file_path, low_memory=False)

    temp_df.columns = temp_df.columns.astype(str).str.strip()

    # Kolon eşleştirmelerini uygula
    applicable_mapping = {old: new for old, new in column_mapping.items() if old in temp_df.columns and new not in temp_df.columns}
    if applicable_mapping:
        temp_df.rename(columns=applicable_mapping, inplace=True)

    if "Fwd Header Length.1" not in temp_df.columns and "Fwd Header Length" in temp_df.columns:
        temp_df["Fwd Header Length.1"] = temp_df["Fwd Header Length"]

    # Sadece BENIGN olmayan (saldırı) kayıtları seçelim ki test eğlenceli olsun
    if "Label" in temp_df.columns:
        # Etiket temizliği
        temp_df["_clean_label"] = temp_df["Label"].astype(str).str.strip()
        attacks = temp_df[~temp_df["_clean_label"].str.contains("benign|normal", case=False, na=False)]
        if not attacks.empty:
            # Her dosyadan rastgele 2'şer örnek alalım
            sampled = attacks.sample(n=min(2, len(attacks)), random_state=42)
            dfs.append(sampled)

if not dfs:
    raise ValueError("Test için geçerli saldırı örneği bulunamadı.")

# Tüm örnekleri tek bir test havuzunda birleştir
sample_df = pd.concat(dfs, ignore_index=True)
true_labels = sample_df["Label"].values if "Label" in sample_df.columns else ["Bilinmiyor"] * len(sample_df)

print(f"\n--- TOPLAM {len(sample_df)} FARKLI YILDAN SEÇİLEN AKIŞ ANALİZ EDİLİYOR ---")

# 3. Modelin beklediği feature matrisini hazırla
for f in features:
    if f not in sample_df.columns:
        sample_df[f] = 0.0

X_test = sample_df[features].copy()
for col in X_test.columns:
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce").astype("float32")

X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test_clean = imputer.transform(X_test).astype("float32")

# 4. Tahmin yap
preds_encoded = model.predict(X_test_clean)
probas = model.predict_proba(X_test_clean)
predicted_labels = label_encoder.inverse_transform(preds_encoded)
confidences = probas.max(axis=1)

# 5. Sonuçları Ekrana Yazdır
for i in range(len(sample_df)):
    print(f"\n[Test Akışı #{i+1}]")
    print(f"  Gerçek Etiket (Ground Truth) : {true_labels[i]}")
    print(f"  Modelin Tahmini             : {predicted_labels[i]}")
    print(f"  Güven Skoru (Confidence)    : %{confidences[i]*100:.2f}")

    if predicted_labels[i] == web_candidate_label:
        print("  Sonraki Katman (Next Layer) : HTTP Payload Analizcisine Yönlendiriliyor (Layer 2)")
    else:
        print("  Sonraki Katman (Next Layer) : MITRE Ajanına / Loglama Sistemine Yönlendiriliyor (Layer 3)")

Model paketi yükleniyor...
Test dosyası yükleniyor: data/Tuesday-WorkingHours.pcap_ISCX.csv
Test dosyası yükleniyor: data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Test dosyası yükleniyor: data/IDS2025.xlsx

--- TOPLAM 6 FARKLI YILDAN SEÇİLEN AKIŞ ANALİZ EDİLİYOR ---

[Test Akışı #1]
  Gerçek Etiket (Ground Truth) : FTP-Patator
  Modelin Tahmini             : FTP-BruteForce
  Güven Skoru (Confidence)    : %100.00
  Sonraki Katman (Next Layer) : MITRE Ajanına / Loglama Sistemine Yönlendiriliyor (Layer 3)

[Test Akışı #2]
  Gerçek Etiket (Ground Truth) : SSH-Patator
  Modelin Tahmini             : FTP-BruteForce
  Güven Skoru (Confidence)    : %40.17
  Sonraki Katman (Next Layer) : MITRE Ajanına / Loglama Sistemine Yönlendiriliyor (Layer 3)

[Test Akışı #3]
  Gerçek Etiket (Ground Truth) : PortScan
  Modelin Tahmini             : PortScan
  Güven Skoru (Confidence)    : %99.83
  Sonraki Katman (Next Layer) : MITRE Ajanına / Loglama Sistemine Yönlendiriliyor (Layer 3)

[Test Ak